In [1]:
"""Based partly on: 
    https://github.com/UnixJunkie/smi2sdf3d/blob/master/smi2sdf.py
    http://rdkit.org/UGM/2012/Ebejer_20110926_RDKit_1stUGM.pdf
    http://pubs.acs.org/doi/abs/10.1021/ci2004658
    https://greglandrum.github.io/rdkit-blog/posts/2024-02-11-more-multithreading.html

    Should try to sample all possible conformations based on dihedral angles sampling based on: https://github.com/dkoes/rdkit-scripts/blob/master/rdallconf.py
"""
# Author: Thua-Phong Lam, Jens Carlsson lab, Uppsala University

from openbabel import openbabel as ob
from rdkit import Chem
from rdkit.Chem import AllChem, rdDistGeom, rdForceFieldHelpers
import os
import subprocess
from pathlib import Path
from MolSanitizer.amsol import run_amsol

def convert(data, inf, otf):
    obConversion = ob.OBConversion()
    obMol = ob.OBMol()
    obConversion.SetInAndOutFormats(inf, otf)
    obConversion.ReadString(obMol, data)
    return obConversion.WriteString(obMol)

def embed_smiles(smiles, name, rmsd=0.25, randomSeed=42, numConfs=50):
    """
    Embed SMILES into multiple conformations, minimize using MMFF94s, and return the MOL2 format.
    
    Args:
    smiles (str): The SMILES string of the molecule.
    name (str): The name of the molecule.
    rmsd (float): The RMSD threshold for pruning conformations.
    randomSeed (int): The random seed for reproducibility.
    numConfs (int): The number of conformations to generate.
    
    Returns:
    str: The lowest energy conformation in MOL2 format.
    """
    params = rdDistGeom.ETKDGv3()
    params.numThreads = 0  # Use all available threads
    params.pruneRmsThresh = rmsd  # Prune conformations that are too similar
    params.randomSeed = randomSeed # For reproducibility

    mol = Chem.MolFromSmiles(smiles)
    mol_H = Chem.AddHs(mol)
    res = Chem.Mol(mol_H) # res = result molecule with conformations
    res.RemoveAllConformers() # An empty conformer list

    conf_energies = []
    
    mp = rdForceFieldHelpers.MMFFGetMoleculeProperties(mol_H, mmffVariant="MMFF94s")
    mp.SetMMFFDielectricConstant(80)

    for cid in rdDistGeom.EmbedMultipleConfs(mol_H, numConfs=numConfs, params=params):
        ff = rdForceFieldHelpers.MMFFGetMoleculeForceField(mol_H, mp, confId=cid)
        ff.Minimize()
        energy = ff.CalcEnergy()
        conformer = mol_H.GetConformer(cid)
        conf_energies.append((energy, conformer))
    
    conf_energies = sorted(conf_energies, key=lambda x: x[0]) # sort by increasing E
    energy, conformer = conf_energies.pop(0) # get the lowest energy conformer
    res.AddConformer(conformer, assignId=True) # add it to the conformations list
    res.SetProp("_Name", name)
    netcharge = sum(atom.GetFormalCharge() for atom in res.GetAtoms())
    return Chem.MolToMolBlock(res), netcharge

In [2]:
smiles = "CSc1cccc(NC(=O)C[N@@H+](C)CC(=O)NC(C)(C)C)c1"
name = "ZINC000007659086.1"

In [3]:
mol, netcharge = embed_smiles(smiles, name)

In [4]:
mol2 = convert(mol, "mol", "mol2")

In [5]:
def write_to_file(content, file):
    with open(file, "w") as f:
        f.write(content)

In [7]:
env = os.environ.copy()
env['LD_LIBRARY_PATH'] = os.path.abspath("libs/extralibs-2")  + ':' + env.get('LD_LIBRARY_PATH', '')

In [8]:
if os.getcwd().split('/')[-1] != 'MolSanitizer': os.chdir("../..")
subprocess.run(f"mkdir -p solv/{name}", shell=True)
write_to_file(mol2, f"solv/{name}/{name}.mol2")
os.chdir(f"solv/{name}")
run_amsol.prepare(f"{name}.mol2", name, netcharge)
run_amsol.run('temp.in-hex', 'temp.o-hex', env)
run_amsol.run('temp.in-wat', 'temp.o-wat', env)
run_amsol.process_output('temp.o-wat', 'temp.o-hex', "temp.mol2", "output")
os.chdir("../..")